# Exam Analysis

Analysis of one exam from the grading dashboard pipeline, configured under `[tool.coursedata.gradescope.grading]`
in `pyproject.toml`. Everything specific to the exam comes from that configuration, the latest raw Gradescope
snapshot, and the blank exams in `data/raw/artifacts/<artifacts>/templates/`. Refresh the data with

```
python -m coursedata.dataset get gradescope grading <exam>
python -m coursedata.dataset process gradescope-grading <exam>
```

The exam needs a multiple-choice part named `MC` and a free-response part named `FR`. Two optional keys in the
exam's table connect it to its templates:

```toml
[tool.coursedata.gradescope.grading.<exam>]
artifacts = "ch6"              # folder under data/raw/artifacts (default: the exam key)

[tool.coursedata.gradescope.grading.<exam>.templates]
"Makeup A" = "ch6-MUA"         # Gradescope version -> template file stem, where the names don't match
```

Without templates the analysis still runs, but it assumes question *n* is the same item on every version.

A third optional table pairs exam questions with homework problems that practice the same skill, for the
"Homework vs exam" section (`get gradescope grading` downloads those assignments' scores too):

```toml
[tool.coursedata.gradescope.grading.<exam>.homework]
Q4 = { assignment = "8551251", question = "5" }   # exam question 4 <-> Gradescope assignment, question 5
```

A student's lecture section is their roster section if `data/processed/sections/latest/sections.json` (from
`process sections`) lists it as type "Lecture". Lecture sections are compared only if the roster has at least two;
a course whose roster sections are recitations has none.

In [ ]:
EXAM = None  # exam key in pyproject.toml; None when only one exam is configured
MIN_VERSION_N = 5  # smallest version included in the across-version test

In [ ]:
import json

import matplotlib.pyplot as plt
from nyu_colors import LIGHT_VIOLET2, MAGENTA, MEDIUM_GRAY2, NYU_VIOLET, TEAL, ULTRA_VIOLET
import pandas as pd
from IPython.display import Markdown, display

from coursedata.config import FIGURES_DIR, PROCESSED_DATA_DIR
from coursedata.dataset.gradescope.grading import load_exam, read_students

exam = load_exam(EXAM)
d = exam.processed_dir
students = read_students(exam)
progress = pd.read_csv(d / "progress.csv", dtype={"number": str})
stats = pd.read_csv(d / "stats.csv")
unmatched = pd.read_csv(d / "unmatched.csv")
history = pd.read_csv(d / "history.csv", parse_dates=["fetched_at"])

sections_file = PROCESSED_DATA_DIR / "sections" / "latest" / "sections.json"
section_types = (
    {s["section"]: s.get("section_type") for s in json.loads(sections_file.read_text())} if sections_file.exists() else {}
)
students["Lecture"] = students["Section"].where(students["Section"].map(section_types) == "Lecture")
lectures = sorted(students["Lecture"].dropna().unique())
display(Markdown(f"**{exam.title}** (`{exam.key}`), roster `{exam.roster}`, lecture sections: {', '.join(lectures) or 'none found'}"))

## Integrity

Completeness of each roster student's exam, and submissions not matched to a student.

In [ ]:
students.groupby("On Roster")["Completeness"].value_counts()

In [ ]:
cols = ["Last Name", "First Name", "Email", "Section", "MC Version", "FR Version"]
students[students["Completeness"] != "complete"][["Completeness", *cols]]

In [ ]:
unmatched

## Grading progress

In [ ]:
progress[progress["version"] == "ALL"][["problem", "graded", "total", "percent", "rate_per_hour", "finish", "status", "responsible", "graders"]]

In [ ]:
history.pivot(index="fetched_at", columns="problem", values="graded").plot(figsize=(10, 5), title="Graded submissions over time");

## Scores

In [ ]:
stats

In [ ]:
graded = students.dropna(subset=["Total Percent"])
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(graded["Total Percent"], bins=20, edgecolor=NYU_VIOLET, color=LIGHT_VIOLET2)
ax.set_xlabel("Percent Score")
ax.set_ylabel("Number of Students")
ax.set_title(f"{exam.title} Score Distribution")
ax.axvline(graded["Total Percent"].mean(), color=MAGENTA, linestyle="dashed", linewidth=1.5, label=f"Mean: {graded['Total Percent'].mean():.1f}%")
ax.axvline(graded["Total Percent"].median(), color=ULTRA_VIOLET, linestyle="dashed", linewidth=1.5, label=f"Median: {graded['Total Percent'].median():.1f}%")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{exam.key}_score_distribution.png")
plt.show()

## Exam templates

Each Gradescope version is matched to a blank exam in the templates folder: by the `templates` table in
`pyproject.toml` if it names the version, otherwise by a template whose version code ends the version's label
(`ch6-16A` for "Section 16 A", `mt1-A` for "A"). Each numbered question is read from the templates with its
printed points and first line of text.

Versions may number the same item differently (for example, options of a check-all-that-apply group shuffled
between versions). So every version's questions are aligned to a reference version's: by identical text, else by
the same text up to numbers (numeric variants of a problem), preferring the same position.

In [ ]:
import re

import numpy as np
import pymupdf

from coursedata.config import RAW_DATA_DIR
from coursedata.dataset.gradescope.grading import GRADING_CONFIG, load_snapshot, snapshots

snap = load_snapshot(exam, snapshots(exam)[-1])
exam_cfg = GRADING_CONFIG[exam.key]
templates_dir = RAW_DATA_DIR / "artifacts" / exam_cfg.get("artifacts", exam.key) / "templates"


def template_questions(path) -> pd.DataFrame:
    """Numbered questions in a blank exam: printed points (if any) and text up to the first option, by number.

    A question in a group headed "25–26. <statement>" has the statement in front of its own text, so items that
    differ only in the shared statement (truth value and reason pairs) can be told apart.
    """
    lines = [line.strip() for page in pymupdf.open(path) for line in page.get_text().splitlines()]
    lines = [line for line in lines if line]
    rows, expected, group = [], 1, (0, 0, "")
    for i, line in enumerate(lines):
        if g := re.match(rf"^{expected}[–-](\d+)\.\s*(.*)$", line):
            stem = [g.group(2)]
            for following in lines[i + 1 : i + 8]:
                if re.match(rf"^{expected}\.", following):
                    break
                stem.append(following)
            group = (expected, int(g.group(1)), " ".join(stem))
            continue
        # Questions are numbered 1, 2, 3, ... in order, so only the next number counts as a question.
        m = re.match(rf"^{expected}\.(?:\s*\((\d+(?:\.\d+)?) points?\))?\s*(.*)$", line)
        if m:
            text = [m.group(2)]
            for following in lines[i + 1 : i + 8]:
                if re.match(rf"^(A\.|\(a\)|{expected + 1}\.)", following):
                    break
                text.append(following)
            if group[0] <= expected <= group[1]:
                text.insert(0, group[2])
            rows.append({"question": expected, "points": float(m.group(1)) if m.group(1) else np.nan,
                         "text": " ".join(t for t in text if t)})
            expected += 1
    return pd.DataFrame(rows, columns=["question", "points", "text"]).set_index("question")


def _code(text: str) -> str:
    return re.sub(r"[^0-9a-z]", "", text.lower())


def template_for(version: str) -> str | None:
    """Stem of the template file for a Gradescope version label, or None."""
    overrides = exam_cfg.get("templates", {})
    if version in overrides:
        return overrides[version]
    hits = [stem for stem in templates if _code(version).endswith(_code(stem.split("-", 1)[-1]))]
    return max(hits, key=len) if hits else None


def _exact(text: str) -> str:
    return re.sub(r"\s+", "", text)


def _pattern(text: str) -> str:
    """Text with numbers replaced by placeholders that keep which ones are equal: {4, 4} -> {#0,#0}."""
    numbers = {}
    return re.sub(r"\d+", lambda m: f"#{numbers.setdefault(m[0], len(numbers))}", _exact(text))


def _shape(text: str) -> str:
    return re.sub(r"\d+", "#", _exact(text))


def _position(text: str) -> str:
    return ""  # matches anything, so a question left over keeps its own number if that is unclaimed


def align(q: pd.DataFrame, ref: pd.DataFrame) -> dict[int, int]:
    """Map each question number of ``q`` to the reference's number for the same item (unmatched: omitted).

    Tries identical text, then text with the same pattern of numbers, then text up to numbers, then position. At
    each step a question keeps its own number if that matches, else takes the only unclaimed match.
    """
    out = {}
    for key in (_exact, _pattern, _shape, _position):
        ref_keys = ref["text"].map(key)
        for n, text in q["text"].items():
            if n in out:
                continue
            free = ref_keys[~ref_keys.index.isin(out.values())]
            if n in free.index and free[n] == key(text):
                out[n] = n
            elif (hits := free.index[free == key(text)]).size == 1:
                out[n] = hits[0]
    return out


templates = {p.name.split(".")[0]: template_questions(p) for p in sorted(templates_dir.glob("*.pdf"))}
versions = snap.versions[["part", "version"]].drop_duplicates()
versions["template"] = versions["version"].map(template_for)
ref_stem = versions["template"].dropna().iloc[0] if versions["template"].notna().any() else None
ref = templates[ref_stem] if ref_stem else pd.DataFrame(columns=["points", "text"])
alignment = {stem: align(templates[stem], ref) for stem in versions["template"].dropna().unique()}

def by_position(stem: str) -> list[int]:
    """Questions aligned only by position: their text differs from the reference's by more than numbers."""
    q = templates[stem]
    return [n for n, m in alignment[stem].items() if _shape(q.at[n, "text"]) != _shape(ref.at[m, "text"])]


has_template = versions["template"].notna()
versions.loc[has_template, "renumbered"] = versions.loc[has_template, "template"].map(
    lambda s: sum(n != m for n, m in alignment[s].items())
)
versions.loc[has_template, "by position"] = versions.loc[has_template, "template"].map(by_position)
versions.loc[has_template, "unaligned"] = versions.loc[has_template, "template"].map(
    lambda s: sorted(set(templates[s].index) - set(alignment[s]))
)
print(f"Templates: {templates_dir} ({len(templates)} files); reference version: {ref_stem}")
versions

Each version above should have a template, and no unaligned questions. Otherwise, add the version to the
`templates` table in `pyproject.toml`, or check the template. Questions listed under "by position" kept their
number because nothing else matched: expected for variants that change more than numbers (a different graph or
function), but worth a look if a version shuffles items. If the templates folder exists, versions without a
template are left out of the item-level analyses below, because their numbering is unknown. If there are no
templates at all, every version is assumed to share one numbering.

## Item-level data

Per-question scores come from the latest raw snapshot, relabeled by the reference version's question numbers:
`Q7` is question 7 on the reference version, whatever its number on the student's version, and `Q30a` is part (a)
of question 30. Point values come from Gradescope. The analyses use the students who have both parts graded.

In [ ]:
HEADER = re.compile(r"^[\d.]+: Question (\d+)(?:\((\w+)\))? \(([\d.]+) pts\)$")

frames, item_max, skipped = [], {}, []
for (part, version), sc in snap.scores.groupby(["part", "version"]):
    stem = template_for(version)
    if templates and stem is None:
        skipped.append(f"{part} {version} ({len(sc)} students)")
        continue
    mapping = alignment.get(stem, {})
    renamed = {}
    for col in sc.columns:
        m = HEADER.match(col)
        if m and sc[col].notna().any():
            label = f"Q{mapping.get(int(m[1]), int(m[1]))}{m[2] or ''}"
            renamed[col] = label
            item_max[(part, label)] = float(m[3])
    if len(set(renamed.values())) < len(renamed):
        raise ValueError(f"{part} {version}: two questions map to the same item; check its template alignment")
    frames.append(sc.set_index("Email")[list(renamed)].rename(columns=renamed).assign(part=part))

item_scores = pd.concat(frames)
if skipped:
    print("Left out (no template):", ", ".join(skipped))


def _order(label: str) -> tuple:
    m = re.match(r"Q(\d+)(\w*)", label)
    return int(m[1]), m[2]


def part_items(part: str) -> tuple[pd.DataFrame, pd.Series]:
    """Points per student (rows) and item (columns) for one part, and each item's maximum."""
    items = item_scores[item_scores["part"] == part].drop(columns="part").dropna(axis=1, how="all")
    items = items[sorted(items.columns, key=_order)]
    items.columns.name = "item"
    return items, pd.Series({c: item_max[(part, c)] for c in items})


mc, mc_max = part_items("MC")
fr, fr_max = part_items("FR")

# Students with both parts graded (and item scores for both), with section and versions alongside.
both = students.dropna(subset=["MC Score", "FR Score"]).set_index("Email")
both = both[both.index.isin(mc.index) & both.index.isin(fr.index)]
mc, fr = mc.loc[both.index], fr.loc[both.index]
exam_total = both["Total Score"]


def question_text(label: str, width: int = 70) -> str:
    n = _order(label)[0]
    text = ref.at[n, "text"] if n in ref.index else ""
    return text if len(text) <= width else text[: width - 1] + "…"


print(f"{len(both)} students with both parts graded; {mc.shape[1]} MC items, {fr.shape[1]} FR items")

### Point values

Gradescope's point values against those printed on the reference template (subparts summed). A mismatch means
either the exam or the Gradescope outline is wrong. Only printed point values can be checked.

In [ ]:
gs_points = pd.concat([mc_max, fr_max]).groupby(lambda c: _order(c)[0]).sum()
points = pd.DataFrame({"Gradescope": gs_points, "printed": ref["points"]}).dropna()
points[points["Gradescope"] != points["printed"]]

## MC item analysis

For each item:

- **p**: mean proportion of the item's points earned (difficulty).
- **r rest of MC**: corrected item–total correlation, against the MC total *excluding* that item.
- **r rest of exam**: the same against the whole exam (MC + FR) excluding the item. This answers "which questions
  go with success overall".
- **D**: discrimination index, p among the top 27% on the whole exam minus p among the bottom 27%.
- **alpha if deleted**: the MC part's reliability (Cronbach's alpha, which is KR-20 for right/wrong items) with
  the item dropped. If it is higher than the overall alpha, the item is adding noise.

Rule-of-thumb flags: p ≥ 0.9 (very easy), p ≤ 0.3 (very hard), r rest of exam < 0.2 (weak discrimination). The
table is sorted with the most predictive items first. An item that is very hard *and* weakly discriminating is
worth checking for a wrong key or a second defensible answer.

In [ ]:
import scipy.stats as ss


def cronbach_alpha(items: pd.DataFrame) -> float:
    k = items.shape[1]
    return k / (k - 1) * (1 - items.var(ddof=1).sum() / items.sum(axis=1).var(ddof=1))


def flags(row) -> str:
    out = []
    if row["p"] >= 0.9:
        out.append("very easy")
    if row["p"] <= 0.3:
        out.append("very hard")
    if row["r rest of exam"] < 0.2:
        out.append("weak discrimination")
    if row["alpha if deleted"] > alpha_mc:
        out.append("lowers alpha")
    return ", ".join(out)


mc_frac = mc / mc_max
mc_total = mc.sum(axis=1)
ranks = exam_total.rank(pct=True)
upper, lower = ranks >= 0.73, ranks <= 0.27
alpha_mc = cronbach_alpha(mc)

with np.errstate(divide="ignore", invalid="ignore"):  # items everyone got right have no correlation
    items = pd.DataFrame(
        {
            "points": mc_max,
            "p": mc_frac.mean(),
            "r rest of MC": {q: mc[q].corr(mc_total - mc[q]) for q in mc},
            "r rest of exam": {q: mc[q].corr(exam_total - mc[q]) for q in mc},
            "D (top-bottom 27%)": mc_frac[upper].mean() - mc_frac[lower].mean(),
            "alpha if deleted": {q: cronbach_alpha(mc.drop(columns=q)) for q in mc},
        }
    )
items["flags"] = items.apply(flags, axis=1)
items["question"] = items.index.map(question_text)
print(f"MC reliability (Cronbach's alpha): {alpha_mc:.3f}")
items.sort_values("r rest of exam", ascending=False).round(3)

### Item difficulty by version

p for each item on each version. A large spread on one item suggests that version's variant (or its key) is harder
than the others. With about 20 students per version, differences of 0.2 are within noise, and more so for small
versions such as makeups.

In [ ]:
by_version = mc_frac.groupby(both["MC Version"]).mean().T
by_version["spread"] = by_version.max(axis=1) - by_version.min(axis=1)
by_version.round(2)

In [ ]:
heat = by_version.drop(columns="spread")
fig, ax = plt.subplots(figsize=(1.3 * heat.shape[1] + 3, 0.35 * heat.shape[0] + 2))
im = ax.imshow(heat, cmap="Purples", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(heat.shape[1]), heat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heat)), heat.index)
for (i, j), p in np.ndenumerate(heat.to_numpy()):
    ax.text(j, i, f"{p:.2f}", ha="center", va="center", fontsize=8, color="white" if p > 0.6 else "black")
fig.colorbar(im, label="Proportion of points")
ax.set_title("MC item difficulty by version")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{exam.key}_mc_item_by_version.png")
plt.show()

## Free-response summary

Points and percent-of-max statistics for each FR item, plus each item's correlation with the rest of the exam. The
second table shows mean percent by version.

In [ ]:
fr_pct = fr / fr_max * 100
fr_summary = pd.DataFrame(
    {
        "max": fr_max,
        "mean": fr.mean(),
        "sd": fr.std(),
        "median": fr.median(),
        "mean %": fr_pct.mean(),
        "% zero": (fr == 0).mean() * 100,
        "% full credit": (fr == fr_max).mean() * 100,
        "r rest of exam": {q: fr[q].corr(exam_total - fr[q]) for q in fr},
    }
)
fr_summary["question"] = fr_summary.index.map(question_text)
alpha_fr = cronbach_alpha(fr)
print(f"FR reliability (Cronbach's alpha): {alpha_fr:.3f}")
fr_summary.round(2)

In [ ]:
fr_pct.groupby(both["FR Version"]).mean().T.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, 1.2 * fr.shape[1] + 2), 5))
ax.boxplot([fr_pct[q] for q in fr_pct], tick_labels=fr_pct.columns, patch_artist=True,
           boxprops={"facecolor": LIGHT_VIOLET2, "edgecolor": NYU_VIOLET},
           medianprops={"color": MAGENTA})
ax.set_ylabel("Percent of points")
ax.set_title("Free-response scores")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{exam.key}_fr_boxplot.png")
plt.show()

## MC vs FR

Correlation between MC percent and FR percent. Both parts are measured with error, so the observed $r$
understates the relationship between the underlying skills. Dividing by $\sqrt{\alpha_{MC}\,\alpha_{FR}}$ (the
classical correction for attenuation) estimates the correlation between the two parts if both were perfectly
reliable.

In [ ]:
mc_pct = both["MC Score"] / both["MC Max"] * 100
fr_pct_total = both["FR Score"] / both["FR Max"] * 100
r, r_p = ss.pearsonr(mc_pct, fr_pct_total)
rho, rho_p = ss.spearmanr(mc_pct, fr_pct_total)
print(f"Pearson r = {r:.3f} (p = {r_p:.2g}), Spearman rho = {rho:.3f} (p = {rho_p:.2g})")
print(f"Corrected for unreliability: r / sqrt(alpha_MC * alpha_FR) = {r / np.sqrt(alpha_mc * alpha_fr):.3f}")

fig, ax = plt.subplots(figsize=(7, 7))
if len(lectures) > 1:
    colors = [NYU_VIOLET, TEAL, MAGENTA, ULTRA_VIOLET]
    for section, color in zip(lectures, colors * len(lectures)):
        s = both["Lecture"] == section
        ax.scatter(mc_pct[s], fr_pct_total[s], color=color, alpha=0.7, label=f"Section {section}")
else:
    ax.scatter(mc_pct, fr_pct_total, color=NYU_VIOLET, alpha=0.6)
slope, intercept = np.polyfit(mc_pct, fr_pct_total, 1)
xs = np.array([0, 100])
ax.plot(xs, intercept + slope * xs, color=MAGENTA, linestyle="dashed", label=f"r = {r:.2f}")
ax.set(xlim=(0, 105), ylim=(0, 105), xlabel="MC percent", ylabel="FR percent", title="MC vs FR")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{exam.key}_mc_vs_fr.png")
plt.show()

## Homework vs exam

For each exam question paired with homework in `pyproject.toml`, students are grouped by how they did on the
homework problem: full credit, partial credit, or not submitted. The table gives each group's mean score on the
exam question (as a percent of its points) and the share who earned full credit on it, plus the Spearman
correlation between homework and exam scores among students who submitted the homework.

High homework scores paired with low exam scores on the same skill suggest the homework score isn't measuring
what the student can do alone under exam conditions. That's a prompt for a conversation about *how* the homework
got done, not a verdict on any one student.

In [ ]:
from coursedata.dataset.gradescope.grading import homework_scores_path

exam_frac = pd.concat([mc / mc_max, fr / fr_max], axis=1)
HW_HEADER = re.compile(r"^([\d.]+): (.*) \(([\d.]+) pts\)$")


def homework_scores(problem) -> tuple[pd.Series, pd.Series, str] | None:
    """Fraction of points on one homework question and submission status, by email, from the newest snapshot
    that has the assignment; and the question's title. None if no snapshot has it."""
    for snapshot_dir in reversed(snapshots(exam)):
        path = homework_scores_path(snapshot_dir, problem.course, problem.assignment)
        if path.exists():
            sc = pd.read_csv(path)
            sc.index = sc["Email"].str.strip().str.casefold()
            matches = [(c, m) for c in sc.columns if (m := HW_HEADER.match(c)) and m[1] == problem.question]
            if not matches:
                raise ValueError(f"No question {problem.question} in {path}")
            col, m = matches[0]
            return sc[col].where(sc["Status"] == "Graded") / float(m[3]), sc["Status"], m[2]
    return None


def homework_group(score: float, status: str) -> str:
    if status == "Missing":
        return "not submitted"
    if pd.isna(score):
        return "not graded"
    return "full credit" if score >= 1 else "partial credit"


GROUPS = ["full credit", "partial credit", "not submitted"]
rows, pairs = [], []
for item, problems in exam.homework.items():
    if item not in exam_frac:
        print(f"{item}: not an item of this exam; check [tool.coursedata.gradescope.grading.{exam.key}.homework]")
        continue
    for problem in problems:
        found = homework_scores(problem)
        if found is None:
            print(f"{item}: no scores for assignment {problem.assignment}; run `get gradescope grading {exam.key}`")
            continue
        hw, status, title = found
        keys = exam_frac.index.str.strip().str.casefold()
        status = status.reindex(keys).fillna("Missing").set_axis(exam_frac.index)
        hw = hw.reindex(keys).set_axis(exam_frac.index)
        group = pd.Series([homework_group(s, t) for s, t in zip(hw, status)], index=exam_frac.index)
        label = f"{item} vs {title} (assignment {problem.assignment})"
        if hw.notna().sum() < 3:  # nothing (or almost nothing) graded yet
            print(f"{label}: homework not graded yet")
            continue
        row = {"exam item": item, "homework": title, "assignment": problem.assignment}
        for g in GROUPS:
            ex = exam_frac.loc[group == g, item]
            row |= {f"n {g}": len(ex), f"exam mean % ({g})": ex.mean() * 100, f"exam full % ({g})": (ex >= 1).mean() * 100}
        submitted = hw.notna()
        rho, p = ss.spearmanr(hw[submitted], exam_frac.loc[submitted, item])
        row |= {"Spearman rho": rho, "p": p}
        rows.append(row)
        pairs.append((label, group, exam_frac[item]))

homework_vs_exam = pd.DataFrame(rows)
if rows:
    display(homework_vs_exam.set_index(["exam item", "homework"]).round(2).T)
else:
    print("No homework is configured or graded for this exam.")

In [ ]:
if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.5 * len(pairs), 4.5), sharey=True, squeeze=False)
    for ax, (label, group, scores) in zip(axes[0], pairs):
        means = [scores[group == g].mean() * 100 for g in GROUPS]
        counts = [(group == g).sum() for g in GROUPS]
        ax.bar(GROUPS, means, color=[NYU_VIOLET, LIGHT_VIOLET2, MEDIUM_GRAY2])
        for i, (m, n) in enumerate(zip(means, counts)):
            if n:
                ax.text(i, m + 2, f"n={n}", ha="center", fontsize=9)
        ax.set_title(label.split(" (assignment")[0], fontsize=10, wrap=True)
        ax.set_xlabel("Homework result")
    axes[0][0].set_ylabel("Mean exam score on item (%)")
    axes[0][0].set_ylim(0, 110)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{exam.key}_homework_vs_exam.png")
    plt.show()

## Lecture sections

Runs only when the roster has at least two lecture sections. Uses every student with both parts graded, grouped
by lecture section (makeup takers included). For two sections, each row gives the mean difference (first − second),
Cohen's d, and p-values from Welch's t-test and the Mann–Whitney U test. For more, it gives p-values from one-way
ANOVA and the Kruskal–Wallis test.

If each section took its own versions, section and version are confounded: a difference could come from the
versions, the students, or the teaching.

In [ ]:
def compare_sections(values: pd.Series, section: pd.Series) -> dict:
    groups = [values[section == s] for s in lectures]
    out = {f"n {s}": len(g) for s, g in zip(lectures, groups)} | {f"mean {s}": g.mean() for s, g in zip(lectures, groups)}
    if len(groups) == 2:
        a, b = groups
        pooled = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
        out |= {
            "difference": a.mean() - b.mean(),
            "Cohen's d": (a.mean() - b.mean()) / pooled,
            "Welch t p": ss.ttest_ind(a, b, equal_var=False).pvalue,
            "Mann-Whitney p": ss.mannwhitneyu(a, b).pvalue,
        }
    else:
        out |= {"ANOVA p": ss.f_oneway(*groups).pvalue, "Kruskal-Wallis p": ss.kruskal(*groups).pvalue}
    return out


if len(lectures) > 1:
    graded_both = students.dropna(subset=["MC Score", "FR Score", "Total Percent"])
    graded_both = graded_both[graded_both["Lecture"].isin(lectures)]
    measures = {
        "Total Percent": graded_both["Total Percent"],
        "MC %": graded_both["MC Score"] / graded_both["MC Max"] * 100,
        "FR %": graded_both["FR Score"] / graded_both["FR Max"] * 100,
    }
    display(pd.DataFrame({m: compare_sections(v, graded_both["Lecture"]) for m, v in measures.items()}).T.round(3))
else:
    print(f"Skipped: {len(lectures)} lecture section(s) on the roster.")

## Versions

Mean scores by MC version, with a Kruskal–Wallis test of whether total percent differs across the versions with
at least `MIN_VERSION_N` students. A significant result suggests the versions were not equally difficult (or that
different groups of students took them).

In [ ]:
pct = both.assign(**{"MC %": mc_pct, "FR %": fr_pct_total})
sizes = pct["MC Version"].value_counts()
tested = sizes.index[sizes >= MIN_VERSION_N]
if len(tested) > 1:
    print(f"Kruskal-Wallis across {len(tested)} versions:",
          ss.kruskal(*(g["Total Percent"] for v, g in pct.groupby("MC Version") if v in tested)))
pct.groupby("MC Version")[["Total Percent", "MC %", "FR %"]].agg(["count", "mean", "std"]).round(1)